In [1]:
# %%
import subprocess
import sys
print("Hello")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "supervision", "lap", "opencv-python"])

Hello


0

In [2]:
# ============================================================
# DIAGNOSTIC — Read a label file to confirm the format
# ============================================================
# %%
from pathlib import Path

VISDRONE_DIR = Path("/kaggle/input/datasets/banuprasadb/visdrone-dataset/VisDrone_Dataset")
TRAIN_DIR = VISDRONE_DIR / "VisDrone2019-DET-train"

# Read the existing visdrone.yaml from the dataset
yaml_in_dataset = VISDRONE_DIR / "visdrone.yaml"
print("=== Existing visdrone.yaml ===")
print(open(yaml_in_dataset).read())

# Read first 5 label files raw
print("\n=== First 3 label files (raw content) ===")
for lbl in sorted((TRAIN_DIR / "labels").glob("*.txt"))[:3]:
    print(f"\n{lbl.name}:")
    lines = open(lbl).readlines()[:5]
    for l in lines:
        print(f"  {l.strip()}")
    print(f"  (total lines: {len(open(lbl).readlines())})")

=== Existing visdrone.yaml ===
# VisDrone Dataset Configuration
path: ./VisDrone_Dataset
train: VisDrone2019-DET-train/images
val: VisDrone2019-DET-val/images
test: VisDrone2019-DET-test-dev/images

#number of classes
nc: 10

# Class names
names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor


=== First 3 label files (raw content) ===

0000002_00005_d_0000014.txt:
  3 0.776042 0.902778 0.077083 0.061111
  3 0.697396 0.829630 0.063542 0.085185
  3 0.652083 0.786111 0.066667 0.094444
  3 0.617188 0.757407 0.063542 0.070370
  3 0.596354 0.719444 0.067708 0.061111
  (total lines: 82)

0000002_00448_d_0000015.txt:
  3 0.131771 0.753704 0.086458 0.077778
  3 0.232813 0.700000 0.063542 0.081481
  3 0.142187 0.883333 0.086458 0.096296
  3 0.126042 0.950926 0.097917 0.098148
  3 0.151562 0.981481 0.080208 0.037037
  (total lines: 85)

0000003_00231_d_0000016.txt:
  3 0.376563 0.419444 0.042708 0.031481
  3 0.420312 0

In [3]:
# ============================================================
# FIX — Convert VisDrone labels → proper YOLO format & retrain
# ============================================================
# %%
import os, shutil, yaml, cv2
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image
from ultralytics import YOLO

VISDRONE_DIR = Path("/kaggle/input/datasets/banuprasadb/visdrone-dataset/VisDrone_Dataset")
TRAIN_DIR = VISDRONE_DIR / "VisDrone2019-DET-train"
VAL_DIR   = VISDRONE_DIR / "VisDrone2019-DET-val"
TEST_DIR  = VISDRONE_DIR / "VisDrone2019-DET-test-dev"

WORK_DIR  = Path("/kaggle/working")
YOLO_DATA = WORK_DIR / "yolo_dataset_v2"    # fresh folder
RESULTS   = WORK_DIR / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

# ── VisDrone category → our class ────────────────────────────
# 0=ignored, 1=pedestrian, 2=people, 3=bicycle,
# 4=car, 5=van, 6=truck, 7=tricycle,
# 8=awning-tricycle, 9=bus, 10=motor, 11=others
VISDRONE_MAP = {1: 0, 2: 0, 4: 1, 5: 1}    # human=0, car=1
CLASS_NAMES  = ["human", "car"]
COLORS       = {0: (0, 255, 120), 1: (0, 120, 255)}


def detect_label_format(lbl_path):
    """Return 'visdrone' or 'yolo' based on first line."""
    with open(lbl_path) as f:
        line = f.readline().strip()
    if not line:
        return "empty"
    parts = line.split(",")
    if len(parts) >= 6:
        return "visdrone"     # comma-separated 8-field original format
    parts = line.split()
    if len(parts) == 5:
        return "yolo"         # space-separated 5-field YOLO format
    return "unknown"


def convert_visdrone_label(lbl_path, img_path):
    """Convert one VisDrone annotation file → YOLO format lines."""
    img = Image.open(img_path)
    W, H = img.size
    yolo_lines = []

    with open(lbl_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(",")
            if len(parts) < 6:
                continue
            x, y, w, h = int(parts[0]), int(parts[1]), int(parts[2]), int(parts[3])
            cat = int(parts[5])

            if cat not in VISDRONE_MAP:
                continue
            if w < 2 or h < 2:
                continue

            cls_id = VISDRONE_MAP[cat]
            cx = (x + w / 2) / W
            cy = (y + h / 2) / H
            nw = w / W
            nh = h / H
            cx, cy, nw, nh = [min(1.0, max(0.0, v)) for v in [cx, cy, nw, nh]]
            yolo_lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")

    return yolo_lines


def convert_yolo_label_remap(lbl_path):
    """
    If labels are already YOLO format but with wrong class IDs,
    keep only class 0 and 1, discard others.
    """
    yolo_lines = []
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls_id = int(parts[0])
            if cls_id not in [0, 1]:
                continue
            yolo_lines.append(line.strip())
    return yolo_lines


def build_yolo_split(split_dir, out_dir, fmt):
    """Copy images and write corrected YOLO labels."""
    img_out = out_dir / "images"
    lbl_out = out_dir / "labels"
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    img_files = sorted(list((split_dir / "images").glob("*.jpg")) +
                       list((split_dir / "images").glob("*.png")))

    converted = skipped = empty = 0
    for img_path in tqdm(img_files, desc=f"Building {out_dir.name}"):
        lbl_src = split_dir / "labels" / (img_path.stem + ".txt")

        if not lbl_src.exists():
            skipped += 1
            continue

        if fmt == "visdrone":
            lines = convert_visdrone_label(lbl_src, img_path)
        else:
            lines = convert_yolo_label_remap(lbl_src)

        # Copy image
        shutil.copy(img_path, img_out / img_path.name)

        # Write label (even empty — YOLO needs the file)
        with open(lbl_out / (img_path.stem + ".txt"), "w") as f:
            f.write("\n".join(lines))

        if len(lines) == 0:
            empty += 1
        converted += 1

    print(f"  Done: {converted} converted | {skipped} skipped | {empty} background images")
    return converted


# ── Detect format from a sample file ─────────────────────────
sample_lbl = sorted((TRAIN_DIR / "labels").glob("*.txt"))[0]
fmt = detect_label_format(sample_lbl)
print(f"Detected label format: '{fmt}'")
print(f"Sample file: {sample_lbl.name}")
print("First 3 lines:")
for line in open(sample_lbl).readlines()[:3]:
    print(f"  {line.strip()}")

# ── Convert all splits ────────────────────────────────────────
print(f"\nConverting using format: {fmt}")
print("\n[Train]")
n_train = build_yolo_split(TRAIN_DIR, YOLO_DATA / "train", fmt)
print("\n[Val]")
n_val = build_yolo_split(VAL_DIR,   YOLO_DATA / "val",   fmt)
print("\n[Test]")
n_test = build_yolo_split(TEST_DIR,  YOLO_DATA / "test",  fmt)

# Verify no more corrupt labels
print("\n=== Verifying converted labels ===")
for split in ["train", "val"]:
    lbl_dir = YOLO_DATA / split / "labels"
    valid = bad = 0
    for lbl in lbl_dir.glob("*.txt"):
        try:
            lines = open(lbl).readlines()
            for line in lines:
                if line.strip():
                    parts = line.strip().split()
                    assert len(parts) == 5
                    assert int(parts[0]) in [0, 1]
                    assert all(0.0 <= float(v) <= 1.0 for v in parts[1:])
            valid += 1
        except Exception:
            bad += 1
    print(f"  {split}: {valid} valid  |  {bad} bad")

Detected label format: 'yolo'
Sample file: 0000002_00005_d_0000014.txt
First 3 lines:
  3 0.776042 0.902778 0.077083 0.061111
  3 0.697396 0.829630 0.063542 0.085185
  3 0.652083 0.786111 0.066667 0.094444

Converting using format: yolo

[Train]


Building train:   0%|          | 0/6471 [00:00<?, ?it/s]

  Done: 6471 converted | 0 skipped | 787 background images

[Val]


Building val:   0%|          | 0/548 [00:00<?, ?it/s]

  Done: 548 converted | 0 skipped | 17 background images

[Test]


Building test:   0%|          | 0/1610 [00:00<?, ?it/s]

  Done: 1610 converted | 0 skipped | 343 background images

=== Verifying converted labels ===
  train: 6471 valid  |  0 bad
  val: 548 valid  |  0 bad


In [4]:
# ── Write YAML ────────────────────────────────────────────────
# %%
yaml_path = YOLO_DATA / "visdrone.yaml"
yaml_cfg = {
    "path"  : str(YOLO_DATA),
    "train" : "train/images",
    "val"   : "val/images",
    "test"  : "test/images",
    "nc"    : 2,
    "names" : CLASS_NAMES
}
with open(yaml_path, "w") as f:
    yaml.dump(yaml_cfg, f, default_flow_style=False)

print("YAML written:")
print(open(yaml_path).read())

# Sanity check: count objects in converted val labels
human_c = car_c = 0
for lbl in (YOLO_DATA / "val" / "labels").glob("*.txt"):
    for line in open(lbl):
        if line.strip():
            cls = int(line.split()[0])
            if cls == 0: human_c += 1
            elif cls == 1: car_c += 1
print(f"Val set objects — humans: {human_c:,}  |  cars: {car_c:,}")
print("✅ If both numbers are large, conversion worked perfectly!")

YAML written:
names:
- human
- car
nc: 2
path: /kaggle/working/yolo_dataset_v2
test: test/images
train: train/images
val: val/images

Val set objects — humans: 8,844  |  cars: 5,125
✅ If both numbers are large, conversion worked perfectly!


In [5]:
# ── Retrain on fixed data ─────────────────────────────────────
# %%
model = YOLO("yolov8s.pt")

results = model.train(
    data          = str(yaml_path),
    epochs        = 50,
    imgsz         = 640,
    batch         = 16,
    optimizer     = "AdamW",
    lr0           = 0.001,
    lrf           = 0.01,
    warmup_epochs = 3,
    mosaic        = 1.0,
    mixup         = 0.15,
    copy_paste    = 0.1,
    degrees       = 10.0,
    scale         = 0.7,
    fliplr        = 0.5,
    hsv_h         = 0.015,
    hsv_s         = 0.7,
    hsv_v         = 0.4,
    patience      = 15,
    save          = True,
    project       = str(WORK_DIR / "runs"),
    name          = "visdrone_v2_fixed",
    exist_ok      = True,
    device        = 0,
    workers       = 4,
    verbose       = True,
)

BEST_WEIGHTS = WORK_DIR / "runs" / "visdrone_v2_fixed" / "weights" / "best.pt"
print(f"\n✅ Done! Best weights → {BEST_WEIGHTS}")
print("\nExpected after fix:")
print("  Steps/epoch : ~400  (not 10)")
print("  Val images  : 548   (not 2)")
print("  Car AP50    : >0.4  (not 0.25)")

Ultralytics 8.4.50 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_dataset_v2/visdrone.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=visdrone_v2_fixed, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, over

In [11]:
# ============================================================

# CELL — Training Metrics Visualisation

# ============================================================

# %% 

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
RESULTS = WORK_DIR / "results"
RESULTS.mkdir(exist_ok=True)
WORK_DIR = Path("/kaggle/working")



csv_path = WORK_DIR / "runs" / "visdrone_v2_fixed" / "results.csv"


df_train = pd.read_csv(csv_path)

df_train.columns = df_train.columns.str.strip()



metric_map = {

    "train/box_loss"      : "Train Box Loss",

    "train/cls_loss"      : "Train Class Loss",

    "val/box_loss"        : "Val Box Loss",

    "val/cls_loss"        : "Val Class Loss",

    "metrics/mAP50(B)"    : "mAP@0.50",

    "metrics/mAP50-95(B)" : "mAP@0.50:0.95",

}

available = {k: v for k, v in metric_map.items() if k in df_train.columns}

epoch_col = df_train["epoch"] if "epoch" in df_train.columns else df_train.index



fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes = axes.flatten()



colors_list = ["#e74c3c", "#e67e22", "#3498db", "#9b59b6", "#2ecc71", "#1abc9c"]

for ax, (col, name), color in zip(axes, available.items(), colors_list):

    ax.plot(epoch_col, df_train[col], color=color, linewidth=2.5)

    ax.fill_between(epoch_col, df_train[col], alpha=0.1, color=color)

    # Mark best epoch

    if "mAP" in name or "Loss" in name:

        if "Loss" in name:

            best_idx = df_train[col].idxmin()

        else:

            best_idx = df_train[col].idxmax()

        best_val = df_train[col].iloc[best_idx]

        best_ep  = epoch_col.iloc[best_idx]

        ax.axvline(best_ep, color="red", linestyle="--", alpha=0.5)

        ax.scatter([best_ep], [best_val], color="red", zorder=5, s=60)

        ax.annotate(f"Best: {best_val:.4f}", xy=(best_ep, best_val),

                    xytext=(8, 4), textcoords="offset points", fontsize=8, color="red")

    ax.set_title(name, fontsize=12, fontweight="bold")

    ax.set_xlabel("Epoch"); ax.grid(alpha=0.3)



for ax in axes[len(available):]:

    ax.axis("off")



plt.suptitle("Training Metrics — YOLOv8s Fixed on VisDrone2019 (548 val images)",

             fontsize=13, fontweight="bold")

plt.tight_layout()

plt.savefig(RESULTS / "training_metrics.png", dpi=120, bbox_inches="tight")

plt.show()

print("Saved → training_metrics.png")



print("\n📈 Final epoch metrics:")

last = df_train.iloc[-1]

for col, name in available.items():

    print(f"  {name:<25}: {last[col]:.4f}")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/runs/visdrone_v2_fixed/results.csv'